# IOAI — 2026 Local Mock Paintings (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-paintings.zip', '/tmp/paint.zip')
    with zipfile.ZipFile('/tmp/paint.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

In [ ]:
root_path = "data"
seed = 42

# Data

In [ ]:
train_df_ = pd.read_csv(f"{root_path}/train.csv")
test_df_ = pd.read_csv(f"{root_path}/test.csv")

# Subtask 1

In [ ]:
def compute_aas(df):
    aas = np.zeros(len(df), dtype=int)
    aas += (df["stroke_density"] > 0.7) * 2
    aas += (df["complexity"] > 0.65) * 2
    aas += df["uses_gold_leaf"]
    aas += df["has_signature"]
    aas += ((df["num_colors"] > 65) & (df["colorfulness"] > 0.7)) * 2
    aas -= ((df["contrast"] < 0.4) | (df["brightness"] < 0.45) | (df["brightness"] > 0.75))
    return aas

test_aas = compute_aas(test_df_)
subtask1 = np.where(test_aas >= 5, "Autentic", "Incert")

In [ ]:
plt.figure(figsize=(5, 2))
plt.hist(subtask1)

# Subtask 2

In [ ]:
cluster_cols = ["painter_style_score", "fake_style_score", "complexity", "stroke_density", 
                "colorfulness", "brightness", "contrast", "num_colors", "complexity_x_stroke"]

combined_raw = pd.concat([train_df_, test_df_], ignore_index=True)
combined_raw = combined_raw[cluster_cols]
X_cluster = StandardScaler().fit_transform(combined_raw)

km = KMeans(n_clusters=5, n_init=20, random_state=seed)
all_labels = km.fit_predict(X_cluster)
subtask2 = all_labels[len(train_df_) :]

In [ ]:
tsne = TSNE(n_components=2, random_state=seed)

xy = tsne.fit_transform(X_cluster[len(train_df_) :])

sns.scatterplot(x=xy[:, 0], y=xy[:, 1], hue=subtask2)

# Subtask 3

In [ ]:
# clusters only for train now - be sure to avoid leakage
cluster_scaler = StandardScaler()
kmeans = KMeans(n_clusters=5, n_init=20, random_state=seed)

X_cluster_train = cluster_scaler.fit_transform(train_df_[cluster_cols])
kmeans.fit(X_cluster_train)


def add_cluster_info(df):
    df = df.copy()
    X_scaled = cluster_scaler.transform(df[cluster_cols])
    labels = kmeans.predict(X_scaled)
    df["cluster_feat"] = labels.astype(str)  # 1st important feature
    return df

In [ ]:
def clean_df(df):
    df = df.copy()
    df = df.drop(["SampleID", "target_price"], axis=1, errors="ignore")

    dims = df["canvas_size"].str.split("x", expand=True).astype(float)
    df["canvas_area"] = dims[0] * dims[1] # 2nd important feature
    df["aspect_ratio"] = dims[0] / dims[1]
    df["dim0"] = dims[0]
    df["dim1"] = dims[1]
    df = df.drop("canvas_size", axis=1)

    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        dummy = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummy], axis=1)
        df = df.drop(col, axis=1)

    return df


train_df_mod = add_cluster_info(train_df_)
train_df_mod["AAS"] = compute_aas(train_df_mod)
train_df = clean_df(train_df_mod)
y = train_df_["target_price"]

In [ ]:
train_df.head()

In [ ]:
train_df.corrwith(y).sort_values(ascending=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    train_df, y, test_size=0.2, random_state=seed
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def evaluate(model):
    cv = cross_val_score(model, X_train, y_train, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
    cv = cv.mean() + cv.std()

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    score = mean_absolute_error(preds, y_test)
    return score, -cv

In [ ]:
lr = Ridge(alpha=0.1, random_state=seed)

evaluate(lr)

In [ ]:
final_scaler = StandardScaler()
X_full = final_scaler.fit_transform(train_df)

final_model = Ridge(alpha=0.1, random_state=seed)
final_model.fit(X_full, y)

In [ ]:
test_df_mod = add_cluster_info(test_df_)
test_df_mod["AAS"] = compute_aas(test_df_mod)
test_df = clean_df(test_df_mod)

test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

test_df_scaled = final_scaler.transform(test_df)

subtask3 = final_model.predict(test_df_scaled)

In [ ]:
plt.figure(figsize=(12, 3))
plt.hist(subtask3)

# Submission

In [ ]:
def build_df(sid, answers):
    return pd.DataFrame(
        {"SampleID": test_df_["SampleID"], "subtaskID": sid, "Answer": answers}
    )

subtasks = [("Task1", subtask1), ("Task2", subtask2), ("Task3", subtask3)]

submission_df = pd.concat(
    [
        build_df(sid, ans)
        for sid, ans in subtasks
    ]
)

In [ ]:
submission_df.head()

In [ ]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)